# Proposed validation — review before it runs

**What this measures:** The target `trainable_params` is the raw field the repo's own MetaMathQA harness writes into `results/*.json`, read from a real TernaryAdapt run at the llama-3.2-3B rank32 protocol and compared against the published LoRA r=32 corpus row (9,174,720) — directly quantifying the "far fewer trainable parameters than standard LoRA" half of the maintainer's claim on the named dataset, with `eval_loss` from the same run guarding the "without losing fit" half. Because the PR registers the method only when `peft.tuners.ternary_adapt` is imported (`register_peft_method(...)` and the dynamic PeftType member live in the new package's `__init__`, and `peft/tuners/__init__.py` does not import it), the harness cannot resolve `peft_type: TERNARY_ADAPT` in the new experiment config without `preimport: [peft.tuners.ternary_adapt]`.

**Target metric:** `trainable_params`

Remyx wrote this test for the change in this PR. **Nothing here has been executed** — there are no outputs, and no result is being claimed.

Edit it if the measurement is wrong, then mention `@remyx validate` again and it will run what you committed. If anything is missing at run time — an import, a dependency, a device — the run reports it and repairs what it can rather than failing silently.

The executable copy lives at ``, which is what `.remyx/validation.yaml` points at; keep the two in step, or point `suite:` here if you would rather maintain the notebook.

## What will run

This validation reuses **the repository's own benchmark** rather than a synthesized stand-in, so the numbers are comparable to the results this repo publishes.

- **runner**: `method_comparison/MetaMathQA/run.py`
- **experiments**: `experiments/ternary_adapt/llama-3.2-3B-rank32`
- **results read from**: `method_comparison/MetaMathQA/results/*.json`
- **comparable rows**: `ternary_adapt`
- **pre-imports before loading the config**: `peft.tuners.ternary_adapt`

## The criteria this is judged against

From `.remyx/validation.yaml` — thresholds live here, not in the test, so a failing measurement reports rather than crashes.

```yaml
# TernaryAdapt vs the published LoRA r=32 row on the repo's OWN MetaMathQA harness. The claim names MetaMathQA
# and this repo publishes comparable numbers from exactly this harness, so suite kind (a): run the real protocol
# (llama-3.2-3B SFT + eval) with ternary_adapt injected through get_peft_model by run.py, and compare to the corpus.
# No baseline arm runs — the published corpus row IS the baseline.
benchmarks:
  - name: ternary-adapt-metamathqa-rank32
    suite:
      harness:
        runner: method_comparison/MetaMathQA/run.py
        experiments: experiments/ternary_adapt/llama-3.2-3B-rank32
        results_glob: "method_comparison/MetaMathQA/results/*.json"
        method: ternary_adapt
        # The PR confines registration to the new package's __init__ (register_peft_method + a dynamically added
        # PeftType member executed on import; peft/tuners/__init__.py does not import it), so without preimport
        # the harness cannot load adapter_config.json with peft_type TERNARY_ADAPT.
        preimport:
          - peft.tuners.ternary_adapt
      scorer: trainable_params
    metrics:
      # TARGET: "far fewer trainable parameters than standard LoRA" — raw harness field, lower is better.
      # Derivation: published LoRA r=32 q/v row = 28 x [32*(3072+3072) + 32*(3072+1024)] = 9,174,720 trainable.
      # TernaryAdapt default near-square Kronecker blocks = 28 x [(64*64+48*48) + (32*64+32*48)] = 279,552
      # (~32.8x fewer). Threshold 1,146,840 = 9,174,720 / 8: demands at least 8x fewer than LoRA while sitting
      # ~4x above the derived ternary value, so a correct implementation clears it and a LoRA-sized mask fails it.
      - name: trainable_params
        direction: min
        threshold: 1146840
        role: target
      # GUARDRAIL: "without losing fit on MetaMathQA" — raw harness field, lower is better, ceiling that must
      # not rise. Anchor: SFT'd llama-3.2-3B on MetaMathQA lands ~0.6-0.9 eval_loss in this corpus (the published
      # lora--llama-3.2-3B-rank32 row at baseline.source is the source of record and clears 1.1), while a
      # ternarized base whose mask fails to recover fit sits >= 1.4, so 1.1 separates retention from degeneration;
      # tighten toward baseline eval_loss * 1.03 once the corpus row is read.
      - name: eval_loss
        direction: min
        threshold: 1.1
        role: guardrail
    baseline:
      source: method_comparison/MetaMathQA/results/lora--llama-3.2-3B-rank32.json
      values:
        # Derived (28-layer q/v LoRA r=32 = 9,174,720); the corpus file at `source` is the source of record and overrides.
        trainable_params: 9174720
        # eval_loss for the no-regression comparison is read from `source` at scoring time.
    policy:
      guardrail_veto: true
    held_constant:
      - "base model meta-llama/Llama-3.2-3B, identical to every llama-3.2-3B-rank32 corpus row"
      - "training budget (epochs, lr, batch, seed) from the shared rank32 training_params protocol, identical to sibling experiment dirs — copy the sibling training_params.json verbatim if it differs"
      - "adapter targets q_proj and v_proj over all 28 decoder layers, matching the lora rank32 row"
      - "same MetaMathQA train/eval split and eval_loss computation as the published corpus rows"
    avoid:
      - "unpinned base-model revision on the Hub; accept only the revision the harness records in the corpus rows"
      - "the PR's synthetic-teacher CPU surrogate (method_comparison/MetaMathQA/eval_ternary_adapt_param_efficiency.py) as a stand-in for the real MetaMathQA protocol — a synthetic teacher is not MetaMathQA and is not comparable to the published corpus"
      - "analytic parameter counts instead of the trainable_params field the harness itself writes"
    compute:
      tier: gpu
      # llama-3.2-3B bf16 SFT on MetaMathQA at the rank32 corpus protocol (a few thousand optimizer steps) plus
      # eval is roughly 2-6 h on one A100; 43200 s (12 h) covers a full-3-epoch worst case for ONE arm.
      timeout_s: 43200
    provenance:
      trainable_params: "user_guidance"
      eval_loss: "user_guidance"
      suite: "repo_runner:method_comparison/MetaMathQA/run.py"
      baseline: "published_corpus:method_comparison/MetaMathQA/results/lora--llama-3.2-3B-rank32.json"
      experiments: "repo_config_shape:method_comparison/MetaMathQA/experiments/adalora/llama-3.2-3B-rank32"
      preimport: "pr_diff:src/peft/tuners/ternary_adapt/__init__.py"
      held_constant: "protocol_doc:method_comparison/README.md"
      eval_loss_threshold: "inferred"
```